In [3]:
import hashlib

In [ ]:
def get_file_checksum(filename):
    h = hashlib.sha256()
    with open(filename, 'rb') as file:
        while True:
            # Читаем блок 4KB
            chunk = file.read(4096)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()[:10]

version = get_file_checksum('Analiz_sklada_HBS.ipynb')

print('version = ', version)

In [5]:
import pandas as pd
import datetime as dt
from dateutil.relativedelta import relativedelta
import re
import json
from tqdm import tqdm
import numpy as np
from io import StringIO
import time
import json
import ast

import sys
sys.path.insert(1, '..')
sys.path.insert(1, '../..')

##################################
import os
from dotenv import dotenv_values

import plotly.express as px
import plotly

config = dotenv_values('../../env.env')
for key,val in config.items():
    os.environ[key] = val
    print(key)

login
password
login_trino
password_trino
net_folder_path
unliquide_net_folder_path
aog_file_name
path_to_vpz
path_to_vpz_alt
path_to_vpz_bazy


In [6]:
from get_data_from_Amos import get_data_with_params_from_dict

Connection to PostgreSQL established successfully!


In [7]:
from my_trino import insert_to_trino

In [24]:
base_path = os.environ['net_folder_path']
path_to_vpz = os.environ['path_to_vpz']
path_to_vpz_alt = os.environ['path_to_vpz_alt']

path_to_vpz = os.environ['path_to_vpz']
path_to_vpz_alt = os.environ['path_to_vpz_alt']

# Для корректной работы с sql запросами.
os.environ['relative_sql_path'] = 'sql/analiz_sklada_HBS/'
os.environ['relative_data_path'] = 'data/analiz_sklada_za_2_goda'

In [9]:
# Текущая дата 
# Конец периода для анализа
date_y         = dt.date.today()
# Срезы склада на 1 и 2 года назад. 
# Внимание! Не все даты есть в слепках складов! Нужно проверять дату на наличие. 
# Середина периода для анализа 
date_y_minus_1 = dt.date(2025,4,1)
# Начало периода для анализа 
date_y_minus_2 = dt.date(2024,4,1)

#Для фильтрации pr и allocation (год вперёд от текущей даты)
date_y_plus_1  = dt.date.today() + relativedelta(years=1)

#Cуффиксы для колонок в табличку 
suf_minus_2_y = f'beg_of_p'
suf_minus_1_y = f'mid_of_p'
suf_y         = f'end_of_p'

date_y ,date_y_minus_1 ,date_y_minus_2 ,date_y_plus_1, suf_y,suf_minus_1_y,suf_minus_2_y

(datetime.date(2026, 5, 14),
 datetime.date(2025, 4, 1),
 datetime.date(2024, 4, 1),
 datetime.date(2027, 5, 14),
 'end_of_p',
 'mid_of_p',
 'beg_of_p')

# Список ВПЗ, HBS

In [10]:
set_hbs = set()

# Забираем данные из БД 

In [11]:
df_part_description = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/part_description.sql',
                                                    dict_with_str_params={})
df_part_description.shape

(274134, 6)

In [13]:
df_hbs_dict =  get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/part_HBS.sql',
                                                    dict_with_str_params={})
set_hbs = set(df_hbs_dict['partno'])

df_hbs_dict.shape

(3804, 1)

In [15]:
df_part_alternate = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/part_alternate.sql',
                                                    dict_with_str_params={})
df_part_alternate.shape

(83861, 2)

In [25]:
%%time
#Текущий склад
df_stock = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/stock.sql',
                                          dict_with_str_params={'date':date_y})

df_stock['max_trx_date'] = df_stock['max_trx_date'].apply(lambda x: pd.to_datetime(x))

df_stock = df_stock[~df_stock['location'].apply(lambda x: 'RECERTIFY PARTS' in str(x) )].copy()

print(df_stock.shape)

(47907, 37)
CPU times: total: 6.94 s
Wall time: 16.6 s


In [ ]:
#todo отфильтровать на экономические проекты # 

In [26]:
df_stock['projectno'].unique()

array([None, 'HBS SBI', 'СТОР РЕМ КОМП', 'BR_S7_DA32B120',
       'ML_S7_DA32B120', 'LM_S7_OVB_20', 'ЗПД', 'COM_S7_OE17120',
       'BCOM_S7_O_120', 'BAC_S7_O_120', 'BAD_S7_O_120', 'TRANSFER PO',
       'SC_S7_DME_20', 'NL_S7_OB73B120', 'NL_S7_DB73H120',
       'NL_S7_DB73B120', 'HBS GLP', 'POR', 'QUAR STORAGE',
       'COM_S7_OA32120', 'BAD_S7_D_120', 'TL_S7_D120', 'SC_SU_DME_20',
       'COM_S7_OB73120', 'MJO', 'ML_S7_OA32B120', 'LOAN 1',
       'RA_S7_DA32120', 'C_VPBDF_D_121', 'LM_S7_DME_20', 'ES_S7_DA32120',
       'ES_S7_OA32120', 'RA_S7_OA32120', 'OB_S7_OA32120', 'C_VPBND_O_122',
       '10_73667_D_123', '10_73669_D_123', 'OV_S7_DA32120',
       '12_73689_O_125', 'LM_S7_DME_19', 'ES_643822_S_26',
       'ES_643968_S_27', 'LM_GH_OVB_19', 'C_02870_O_123', 'OM_S7_OB73120',
       'BCOM_S7_M_120', 'ES_S7_OB73120', 'CM_S7_MA32120',
       '6Y_73490_D_125', 'CLM_S7_DME_20', 'SALE_MSN 2369',
       'COM_S7_DA32120', 'COM_S7_DB73120', 'OB_S7_OB73120',
       'OB_S7_DB73120', 'COM_S7_DE1

In [27]:
# Забираем все выдачи АТИ по pickslip_booked 
# за 2 года. 
df_ps_booked = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/pickslip.sql',
                                                 dict_with_str_params={'date':date_y,
                                                                       'date_minus_2_y':date_y_minus_2})
df_ps_booked.shape

(1625661, 33)

In [ ]:
#todo как мы отфильтровываем выдачу на ВС , из какого эк проекта нас интересует 

In [28]:
#todo отфильтровать склад окончательно на HBS + посчитать стоимость HBS на дату 

In [29]:
# Аллокация 
df_al_c = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/stock_to_part_request_C.sql',
                                              dict_with_str_params={})
df_al_c.shape

(130593, 30)

In [30]:
# Аллокация 
df_al_r = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/stock_to_part_request_R.sql',
                                              dict_with_str_params={})
df_al_r.shape

(8717, 30)

In [31]:
# PR вместе без аллокации  
df_pr = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/part_request.sql',
                                       dict_with_str_params={})
df_pr.shape

(325861, 30)

## Заказы к pr в состоянии Draft, для исключения фильтрации Ordered pr, если заказ в состоянии Draft    

In [32]:
%%time  
df_pr_to_order = get_data_with_params_from_dict(path_to_sql=f'{os.environ['relative_sql_path']}/pr_to_order_allocation.sql',
                                                dict_with_str_params={})
print(df_pr_to_order.shape)

(236, 34)
CPU times: total: 31.2 ms
Wall time: 387 ms


### Фильтруем pr: убираем все 'Ordered' pr, оставляем их только если заказ не в состоянии 'Draft'.  

In [35]:
%%time 

df_pr = df_pr[(df_pr['availability_status']!='O')|
              ((df_pr['availability_status']=='O')&
               (df_pr['part_requestno_i'].isin(df_pr_to_order['part_requestno_i'])))].copy()

print(df_pr.shape)

(321786, 30)
CPU times: total: 844 ms
Wall time: 857 ms


# todo выгрузить ЗАКАЗЫ !!!!!!

# Группировка данных 

## Взаимозаменяемость 

In [37]:
df_part_alternate_pn = df_part_alternate.groupby(['partno']).agg({'altpartno':lambda x: set(x)}).reset_index()

## Аллокация 

In [39]:
def get_new_valid_due_date(dict_):
    # date_y        - текущая дата 
    # date_y_plus_1 - год вперёд от текущей даты  
    if (date_y - dt.timedelta(days=365))<=dict_.get('pf_valid_due_date') and dict_.get('pf_valid_due_date')<=date_y_plus_1:
        return dict_.get('pf_valid_due_date')
    if dict_.get('wp_h_end_date') is not None:
        if (date_y - dt.timedelta(days=365))<=dict_.get('wp_h_end_date') and dict_.get('wp_h_end_date')<=date_y_plus_1:
            return dict_.get('wp_h_end_date')

In [40]:
df_al_c['NEW_valid_due_date'] = df_al_c.apply(lambda row: get_new_valid_due_date(dict(row)) , axis=1)
df_al_r['NEW_valid_due_date'] = df_al_r.apply(lambda row: get_new_valid_due_date(dict(row)) , axis=1)

df_pr['NEW_valid_due_date'] = df_pr.apply(lambda row: get_new_valid_due_date(dict(row)) , axis=1)

In [41]:
len(df_al_c[df_al_c['NEW_valid_due_date'].isna()])/len(df_al_c)

0.23649812777101376

In [42]:
len(df_al_r[df_al_r['NEW_valid_due_date'].isna()])/len(df_al_r)

0.2993002179648962

In [43]:
len(df_pr[df_pr['NEW_valid_due_date'].isna()])/len(df_pr)

0.19305687630909985

In [44]:
df_al_c_pn = df_al_c[(~df_al_c['NEW_valid_due_date'].isna())|
                     (df_al_c['link_type']=='H')].groupby(['consumables_partno'])[[
                 'al_qty']].sum().reset_index().rename({'consumables_partno':'partno'},axis=1)

In [45]:
df_al_r_pn = df_al_r[(~df_al_r['NEW_valid_due_date'].isna())|
                     (df_al_r['link_type']=='H')].groupby(['rotables_partno'])[[
                'al_qty']].sum().reset_index().rename({'rotables_partno':'partno'},axis=1)

In [46]:
df_pr_pn = df_pr[~df_pr['NEW_valid_due_date'
               ].isna()].groupby(['pr_partno'])[[
                'qty']].sum().reset_index().rename({'pr_partno':'partno',
                                                     'qty':'qty_pr'},axis=1)

In [47]:
df_al = pd.concat([df_al_c_pn,df_al_r_pn])
df_al.rename({'al_qty':'qty_allocation'},axis=1,inplace=True)

## Текущие остатки на складе 

In [48]:
df_stock_pn = df_stock.groupby(['partno']).agg({'qty':'sum',
                                                                                'average_price':'median',
                                                                                'sn_bn':lambda x: set(x),
                                                                                'projectno':lambda x: list(x),
                                                                                'project_description':lambda x: list(x)}).reset_index().rename(
    {'qty':f'qty_{suf_y}',
     'average_price':f'average_price_{suf_y}',
     'sn_bn':f'sn_bn_set_{suf_y}'}, axis=1)



df_stock_pn

,partno,qty_end_of_p,average_price_end_of_p,sn_bn_set_end_of_p,projectno,project_description
0,0-111-0003-2000,2.8,6988.45,"{1001535422, 0004180679}","[6Y_73437_M_124, ЗПД]","[6Y_Check RA-73437 АК Сибирь, ЗАКУПКА ПОД ДЕФЕ..."
1,0-111-0006-2000,2.0,2381.23,{N/A},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ]
2,0-111-0016-2000TU,3.0,8023.18,{2026M139},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ]
3,0-111-0017-2000,1.0,43202.61,{N/A},[BM_73310_D_124],[БАЗОВОЕ ТО RA-73310_А321_АКС]
4,0-111-0021-2000,1.0,23029.99,{N/A},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ]
...,...,...,...,...,...,...
19681,С7И23627П.25.К2-6,1.0,26724.00,{С7И/058/26},[21G_S7_OVB_20],[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ]
19682,С7И23627П.25.К2-7,1.0,30261.00,{С7И/059/26},[21G_S7_OVB_20],[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ]
19683,С7И24089И.25.К1,493.0,8733.61,"{УППК/02/25, УППК/02/26}","[21G_S7_OVB_20, 21G_S7_OVB_20, 21G_S7_OVB_20, ...","[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ, РАБОТЫ ПО P..."
19684,С7И24576И.25.K1,16.0,148103.77,"{УПМК/11/2025, УПМК/12/2025, УПМК/01/2026}","[21G_S7_OVB_20, 21G_S7_OVB_20, 21G_S7_OVB_20, ...","[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ, РАБОТЫ ПО P..."


## PS

In [58]:
df_ps_booked.iloc[0]

mat_class                                        C
projectno_i                                22040.0
consignment_owner                             None
picking_listno_i                              None
pickslipseqno_i                            5402304
pickslipno                                 2337050
recno                                            0
partno               BMS9-3 TY H-2 CL 7 STYLE 1581
serialno                                          
batchno                       31M0036782/31R000272
expire_date                                  19678
recdetailno_i                              2253154
station                                        OVB
store                                      O_HANG1
location_from                           S-S.P.10.0
location_to                                       
qty_booked                                     0.5
qty_sys                                       13.9
qty_canceled                                   0.0
booking_text                   

In [52]:
# Расчет потребления через pickslip_booked 
df_ps_booked['qty'] = df_ps_booked['qty_booked'] - df_ps_booked['qty_canceled']
df_h_pn = df_ps_booked.groupby(['partno']).agg({'qty':'sum'}).reset_index()

In [53]:
df_h_pn.columns = ['partno',
                   'qty']

In [54]:
df_h_pn

,partno,qty
0,"""ВИБРАГАРД"" (7-111) РАЗМЕР 10",21.00
1,"""ВИБРАГАРД"" (7-111) РАЗМЕР 9",4.00
2,- НАКОНЕЧНИК DN02512 BLUE,553.00
3,/ ПЛЕНКА ROXELPRO (ROXONE),4.00
4,0-111-0002-2000,1.00
...,...,...
45687,ЭТИКЕТКИ VELL 1349591,5.00
45688,ЭТИЛАЦЕТАТ (КГ),3.88
45689,ЮЛИЯ-209,905.00
45690,ЮЛИЯ-219,16233.00


In [ ]:
## Учет текущего состояния stock Rotables, в т.ч. что установлено на ВС / engine 

In [ ]:
# Обогащенеи на аллоцированное кол-во на складе. 
df_stock_pn = df_stock_pn.merge(df_al, how='left')

In [60]:
# Обогащенеи на кол-во pr. 
df_stock_pn = df_stock_pn.merge(df_pr_pn, how='left')

In [62]:
df_stock_pn

,partno,qty_end_of_p,average_price_end_of_p,sn_bn_set_end_of_p,projectno,project_description,qty_pr
0,0-111-0003-2000,2.8,6988.45,"{1001535422, 0004180679}","[6Y_73437_M_124, ЗПД]","[6Y_Check RA-73437 АК Сибирь, ЗАКУПКА ПОД ДЕФЕ...",NaN
1,0-111-0006-2000,2.0,2381.23,{N/A},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ],NaN
2,0-111-0016-2000TU,3.0,8023.18,{2026M139},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ],NaN
3,0-111-0017-2000,1.0,43202.61,{N/A},[BM_73310_D_124],[БАЗОВОЕ ТО RA-73310_А321_АКС],NaN
4,0-111-0021-2000,1.0,23029.99,{N/A},[ЗПД],[ЗАКУПКА ПОД ДЕФЕКТЫ],NaN
...,...,...,...,...,...,...,...
19681,С7И23627П.25.К2-6,1.0,26724.00,{С7И/058/26},[21G_S7_OVB_20],[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ],1.0
19682,С7И23627П.25.К2-7,1.0,30261.00,{С7И/059/26},[21G_S7_OVB_20],[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ],1.0
19683,С7И24089И.25.К1,493.0,8733.61,"{УППК/02/25, УППК/02/26}","[21G_S7_OVB_20, 21G_S7_OVB_20, 21G_S7_OVB_20, ...","[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ, РАБОТЫ ПО P...",NaN
19684,С7И24576И.25.K1,16.0,148103.77,"{УПМК/11/2025, УПМК/12/2025, УПМК/01/2026}","[21G_S7_OVB_20, 21G_S7_OVB_20, 21G_S7_OVB_20, ...","[РАБОТЫ ПО PART 21G OVB АК СИБИРЬ, РАБОТЫ ПО P...",NaN


In [ ]:
#todo повторить все расчеты с учетом альтернативности 

In [ ]:
%%time
df_stock_pn['qty_pr_alt'] = df_stock_pn['altpartno'].apply(lambda set_alternatives: 
                                df_pr[df_pr['pr_partno'].isin(set_alternatives)]['qty'].sum() if len(set_alternatives)>0 else 0)
df_stock_pn['qty_pr_w_alt'] = df_stock_pn['qty_pr'] + df_stock_pn['qty_pr_alt'] 